# 第65课 · 🎨 训练可视化 — Loss / Acc 曲线，梯度（gradient）范数（gradient norm），权重分布（weight distribution）直方图

**目标**：用图形手段把训练过程看穿——从 `Value` 节点组成的 DAG，到 train/val loss 曲线的形态，再到关键词识别的混淆矩阵（confusion matrix，CM）错误模式。

🔗 **Aurora 连接**：
- `Value` 计算图（computation graph） → `L54_value_autograd`、`L55_forward_pass`
- 训练循环 → `L58_training_loop`
- 特征提取 → `src/aurora/audio/mel.py` mel 滤波器组（mel filterbank）
- 模型骨干 → L58 中定义的 KeywordCNN


← **上一课**　[L64 · 训练评估闭环](L64_kws_train_eval.ipynb)

> 上节课学习了 **训练评估闭环**：train loop + val loop，准确率 / 混淆矩阵 / 过拟合诊断。  
> 本课将探讨 **训练可视化**。

可视化不是装饰——它是调试工具。计算图告诉你梯度从哪里流、在哪里断；损失曲线告诉你模型是欠拟合（underfitting）还是记住了训练集；混淆矩阵告诉你哪对关键词被互相混淆，指向特征层的具体弱点。三张图合在一起，基本能定位训练失败的根因。

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import itertools

## 1. Value 计算图的 DAG 结构

每次写 `c = a + b`，背后创建了一张有向无环图（DAG）：

```
a ──┐
    ├─[+]── c ──┐
b ──┘            ├─[*]── e
                 │
d ───────────────┘
```

- **节点**：每个 `Value` 对象，存 `data`（前向值）和 `grad`（反向梯度）
- **边**：算子（`+`、`*`、`**`），方向是数据流方向
- **`_prev`**：每个节点记录自己的直接输入节点集合；反向传播（backpropagation）沿 `_prev` 回溯

用 matplotlib 绘制 DAG：把节点画成圆框，边画成箭头，框内写 `data` 和 `grad`，框外写算子名称。不依赖 graphviz，纯 matplotlib 即可，在任何环境都能跑。

In [ ]:
# ── 演示：用 matplotlib 画 Value 计算图的 DAG ──
# 小图：L = (a * b) + b**2，a=2, b=3
# L = 6 + 9 = 15；dL/da = b = 3；dL/db = a + 2b = 2+6 = 8

nodes = {
    'a':    {'data':  2.0, 'grad': 3.0, 'op': ''},
    'b':    {'data':  3.0, 'grad': 8.0, 'op': ''},
    'a*b':  {'data':  6.0, 'grad': 1.0, 'op': '*'},
    'b**2': {'data':  9.0, 'grad': 1.0, 'op': '**2'},
    'L':    {'data': 15.0, 'grad': 1.0, 'op': '+'},
}
edges = [
    ('a',    'a*b'),
    ('b',    'a*b'),
    ('b',    'b**2'),
    ('a*b',  'L'),
    ('b**2', 'L'),
]
pos = {
    'a':    (0, 2),
    'b':    (0, 0),
    'a*b':  (2, 2),
    'b**2': (2, 0),
    'L':    (4, 1),
}

fig, ax = plt.subplots(figsize=(9, 4))
ax.set_xlim(-0.8, 5.2)
ax.set_ylim(-0.8, 3.2)
ax.axis('off')
ax.set_title('L = (a * b) + b**2  的计算图（前向值 | 梯度）', fontsize=13)

BOX_W, BOX_H = 1.1, 0.7
leaf_color = '#d4e6f1'
op_color   = '#fdebd0'
for name, (x, y) in pos.items():
    is_leaf = nodes[name]['op'] == ''
    fc = leaf_color if is_leaf else op_color
    rect = mpatches.FancyBboxPatch(
        (x - BOX_W/2, y - BOX_H/2), BOX_W, BOX_H,
        boxstyle='round,pad=0.05', facecolor=fc, edgecolor='#555', linewidth=1.5
    )
    ax.add_patch(rect)
    d  = nodes[name]['data']
    g  = nodes[name]['grad']
    op = nodes[name]['op']
    label = f'{name}\ndata={d:.1f} | grad={g:.1f}'
    ax.text(x, y + 0.05, label, ha='center', va='center', fontsize=8.5)
    if op:
        ax.text(x, y - BOX_H/2 - 0.18, op, ha='center', va='top',
                fontsize=9, color='#c0392b', fontweight='bold')

for src, dst in edges:
    x0, y0 = pos[src]
    x1, y1 = pos[dst]
    dx, dy = x1 - x0, y1 - y0
    shrink = BOX_W / 2 + 0.05
    length = (dx**2 + dy**2) ** 0.5
    ux, uy = dx / length, dy / length
    ax.annotate('',
        xy=(x1 - ux * shrink, y1 - uy * shrink),
        xytext=(x0 + ux * shrink, y0 + uy * shrink),
        arrowprops=dict(arrowstyle='->', color='#2c3e50', lw=1.4)
    )

ax.add_patch(mpatches.Patch(facecolor=leaf_color, edgecolor='#555', label='叶节点（输入变量）'))
ax.add_patch(mpatches.Patch(facecolor=op_color,  edgecolor='#555', label='算子节点'))
ax.legend(loc='lower right', fontsize=9)
plt.tight_layout()
plt.show()
print('✅ DAG 图绘制完成；蓝色=叶节点，橙色=算子节点；grad 已由 backward() 填入')

## 2. 损失曲线：识别过拟合（overfitting）的早期信号

训练过程中最重要的两条曲线：

- `train_loss`：模型在训练集上的均方误差（或交叉熵），**应该单调下降**
- `val_loss`：在验证集上的同一指标，**初期跟随下降，过拟合后开始上翘**

两者的差距叫**泛化间隔**（generalization gap）：

```
gap(epoch) = val_loss(epoch) - train_loss(epoch)
```

**早期停止（early stopping）的信号**：当 `val_loss` 连续 `patience` 个 epoch 没有下降时，认为过拟合已开始，停止训练并取验证最优的权重。

对于关键词识别（KeywordCNN），过拟合常见原因：
- 训练集声学环境单一，验证集有噪声
- 模型容量（卷积（convolution）核数）远大于训练样本量
- Mel 特征维度（`n_mels=40`）相对样本数过高

In [ ]:
# ── 演示：生成典型的 train/val loss 曲线（含过拟合阶段）──
np.random.seed(42)
epochs = np.arange(1, 61)

train_loss = 1.0 * np.exp(-0.08 * epochs) + 0.05 + 0.01 * np.random.randn(60)
val_decay  = 1.0 * np.exp(-0.05 * epochs) + 0.15
overfit    = np.where(epochs > 25, 0.006 * (epochs - 25), 0.0)
val_loss   = val_decay + overfit + 0.015 * np.random.randn(60)

best_epoch = int(np.argmin(val_loss)) + 1

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(epochs, train_loss, label='train loss', color='#2980b9', lw=2)
ax.plot(epochs, val_loss,   label='val loss',   color='#e74c3c', lw=2)
ax.axvline(best_epoch, color='#27ae60', ls='--', lw=1.5,
           label=f'early stop @ epoch {best_epoch}')
ax.fill_between(epochs,
    np.minimum(train_loss, val_loss),
    np.maximum(train_loss, val_loss),
    alpha=0.12, color='#8e44ad', label='泛化间隔')
ax.set_xlabel('Epoch', fontsize=11)
ax.set_ylabel('Loss', fontsize=11)
ax.set_title('Train vs Val Loss — 过拟合早期信号', fontsize=13)
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

gap_at_best = val_loss[best_epoch-1] - train_loss[best_epoch-1]
print(f'✅ best epoch = {best_epoch}')
print(f'   val_loss_min  = {val_loss[best_epoch-1]:.4f}')
print(f'   泛化间隔 gap  = {gap_at_best:.4f}')
print('   → 紫色区域越大，过拟合越严重；早停在绿线处保存最优权重')

In [ ]:
# 可视化梯度范数（gradient norm visualization）
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

# 演示：定义小型模型，运行一次前向 + 反向，填充梯度
model = nn.Sequential(
    nn.Linear(40, 64), nn.ReLU(),
    nn.Linear(64, 32), nn.ReLU(),
    nn.Linear(32, 10)
)
x_demo = torch.randn(16, 40)
y_demo = torch.randint(0, 10, (16,))
loss = nn.CrossEntropyLoss()(model(x_demo), y_demo)
loss.backward()

grad_norms = []
for name, param in model.named_parameters():
    if param.grad is not None:
        grad_norms.append((name, param.grad.norm().item()))

names = [n for n, _ in grad_norms]  # 使用完整参数路径，区分不同层
norms = [v for _, v in grad_norms]

plt.figure(figsize=(10, 4))
plt.bar(range(len(norms)), norms)
plt.xticks(range(len(names)), names, rotation=45, ha='right')
plt.ylabel('梯度 L2 范数（gradient L2 norm）')
plt.title('各参数梯度范数')
plt.tight_layout()
plt.show()

In [ ]:
# 可视化权重分布（weight distribution histograms）
fig, axes = plt.subplots(2, 3, figsize=(12, 6))
axes = axes.flatten()

params_to_plot = [(name, param) for name, param in model.named_parameters() 
                  if param.requires_grad and param.dim() >= 2][:6]

for i, (name, param) in enumerate(params_to_plot):
    axes[i].hist(param.detach().cpu().numpy().flatten(), bins=50, edgecolor='none')
    axes[i].set_title(name.split('.')[-1])
    axes[i].set_xlabel('权重值')
    axes[i].set_ylabel('频次')

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('权重分布直方图（weight distribution histograms）')
plt.tight_layout()
plt.show()

## ✏️ 练习：实现 early_stopping()

给定验证集损失列表和 patience 参数，找到**触发早停的 epoch**（从 1 开始计数）。

规则：当最近连续 `patience` 个 epoch 的 val_loss 都**高于**当前最小值时，返回当前最小值对应的 epoch 编号（最优 epoch）。

提示：用一个变量追踪历史最小值和计数器。


In [ ]:
def early_stopping(val_losses: list, patience: int) -> int:
    """
    返回触发早停时的最优 epoch（1-based）。
    若整个序列都没有触发，返回 val_loss 最小的 epoch。

    参数
    ----
    val_losses : list[float]  每个 epoch 的验证集损失
    patience   : int          连续多少个 epoch 无改善则停止

    返回
    ----
    int  最优 epoch 编号（1-based）
    """
    # ── TODO：补全实现 ──
    best_epoch = 1
    best_loss  = float('inf')
    no_improve = 0
    for epoch, loss in enumerate(val_losses, start=1):
        raise NotImplementedError("请删除这行，补全实现")
    return best_epoch


# ── 验证 ──
try:
    losses_a = [0.9, 0.7, 0.5, 0.6, 0.65, 0.7, 0.72, 0.75]  # 从 epoch 3 开始上升，patience=3 → 停在 epoch 3
    result_a = early_stopping(losses_a, patience=3)
    assert result_a == 3, f'期望 3，得到 {result_a}'

    losses_b = [0.9, 0.8, 0.7, 0.6, 0.5, 0.4]  # 一直下降，返回最后一个 epoch
    result_b = early_stopping(losses_b, patience=2)
    assert result_b == 6, f'期望 6，得到 {result_b}'

    losses_c = [1.0, 0.5, 0.6, 0.7, 0.8, 0.9]  # patience=2，epoch 2 最优
    result_c = early_stopping(losses_c, patience=2)
    assert result_c == 2, f'期望 2，得到 {result_c}'

    print('✅ 所有 assert 通过！early_stopping() 实现正确。')
    print(f'   losses_a(patience=3) → best_epoch = {result_a}')
    print(f'   losses_b(patience=2) → best_epoch = {result_b}')
    print(f'   losses_c(patience=2) → best_epoch = {result_c}')
except NotImplementedError:
    print('⚠️  请先补全 early_stopping() 中 TODO 部分，再运行验证。')
except AssertionError as e:
    print(f'❌ 验证失败：{e}')
    print('   提示：检查 best_epoch 的更新条件，以及 no_improve 计数器的重置时机。')


## 过渡说明：mel 谱误分类对比

以上三类图（DAG 图、损失曲线、梯度范数/权重分布）已演示了训练过程的调试工具链。
接下来通过**混淆矩阵**和 **mel 谱对比**进一步说明：当模型把 "no" 误判为 "yes" 时，频谱上有哪些具体特征让分类器失误。


## 3. 混淆矩阵（Confusion Matrix）— 手工计算与可视化

混淆矩阵 `CM[i, j]` = 真实类别为 `i`、被预测为 `j` 的样本数。
- **对角线**：分类正确；**非对角**：误分类
- **召回率（Recall）** = `CM[i,i] / sum(CM[i,:])` — 该类被正确识别的比例
- **精确率（Precision）** = `CM[i,i] / sum(CM[:,i])` — 预测为该类时真的是该类的比例

下面用 `np.zeros` 手工累积，**不使用 sklearn**。


In [ ]:
# ── 混淆矩阵：手工计算（np only）+ imshow 可视化 ──
# 模拟 10 类关键词分类器在 200 个测试样本上的预测结果
np.random.seed(7)
CLASSES = ['yes','no','up','down','left','right','on','off','stop','go']
N_CLASSES = len(CLASSES)
N_SAMPLES = 200

y_true = np.random.randint(0, N_CLASSES, N_SAMPLES)
# 给分类器加入系统性混淆：'no'(1) 有 30% 概率被误判为 'yes'(0)
y_pred = y_true.copy()
for idx in range(N_SAMPLES):
    if y_true[idx] == 1 and np.random.rand() < 0.30:
        y_pred[idx] = 0  # no → yes 误分类
    elif y_true[idx] != 1 and np.random.rand() < 0.10:
        y_pred[idx] = np.random.randint(0, N_CLASSES)

# 手工累积混淆矩阵
CM = np.zeros((N_CLASSES, N_CLASSES), dtype=int)
for t, p in zip(y_true, y_pred):
    CM[t, p] += 1

# 归一化（按行：每行总和为 1，方便读召回率）
CM_norm = CM.astype(float) / (CM.sum(axis=1, keepdims=True) + 1e-9)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 左：计数矩阵
im0 = axes[0].imshow(CM, cmap='Blues')
axes[0].set_title('混淆矩阵（计数）', fontsize=12)
axes[0].set_xticks(range(N_CLASSES)); axes[0].set_xticklabels(CLASSES, rotation=45, ha='right')
axes[0].set_yticks(range(N_CLASSES)); axes[0].set_yticklabels(CLASSES)
axes[0].set_xlabel('预测类别'); axes[0].set_ylabel('真实类别')
for i in range(N_CLASSES):
    for j in range(N_CLASSES):
        axes[0].text(j, i, CM[i,j], ha='center', va='center', fontsize=8,
                     color='white' if CM[i,j] > CM.max()*0.6 else 'black')
plt.colorbar(im0, ax=axes[0])

# 右：归一化矩阵（召回率视角）
im1 = axes[1].imshow(CM_norm, cmap='Reds', vmin=0, vmax=1)
axes[1].set_title('混淆矩阵（行归一化 = 召回率）', fontsize=12)
axes[1].set_xticks(range(N_CLASSES)); axes[1].set_xticklabels(CLASSES, rotation=45, ha='right')
axes[1].set_yticks(range(N_CLASSES)); axes[1].set_yticklabels(CLASSES)
axes[1].set_xlabel('预测类别'); axes[1].set_ylabel('真实类别')
for i in range(N_CLASSES):
    for j in range(N_CLASSES):
        axes[1].text(j, i, f'{CM_norm[i,j]:.2f}', ha='center', va='center', fontsize=7,
                     color='white' if CM_norm[i,j] > 0.6 else 'black')
plt.colorbar(im1, ax=axes[1])

plt.suptitle('KeywordCNN 混淆矩阵 — 模拟数据（no→yes 误分类率=30%）', fontsize=13)
plt.tight_layout()
plt.show()

# 打印 recall 和 precision
recall    = np.diag(CM) / (CM.sum(axis=1) + 1e-9)
precision = np.diag(CM) / (CM.sum(axis=0) + 1e-9)
print('📊 各类 Recall / Precision:')
for c, r, p in zip(CLASSES, recall, precision):
    print(f'   {c:6s}  recall={r:.2f}  precision={p:.2f}')
print()
print('✅ CM["no","yes"] 特别高 → 说明"no"频繁被误判为"yes"')
print('   → 结合下方 mel 谱对比可定位根因：鼻音区能量不足时频谱与"yes"相似')


In [ ]:
# ── 参数实验：合成关键词 mel 图，模拟误分类对比 ──
# 不依赖真实音频；用正弦叠加模拟 yes/no 的粗略频谱特征

SR = 16000
N_MELS = 40      # ✏️ 尝试 20 / 40 / 80，观察频率分辨率变化
HOP    = 160     # ✏️ 尝试 80 / 160 / 320，观察时间分辨率变化
N      = int(SR * 0.5)  # 0.5 秒

def synth_keyword(freqs, amps, n=N, sr=SR):
    t = np.linspace(0, n / sr, n)
    sig = sum(a * np.sin(2 * np.pi * f * t) for f, a in zip(freqs, amps))
    return sig / (np.abs(sig).max() + 1e-9)

def naive_mel(sig, sr=SR, n_mels=N_MELS, hop=HOP, n_fft=512):
    frames = [sig[i:i+n_fft] * np.hanning(n_fft)
              for i in range(0, len(sig) - n_fft, hop)]
    specs  = np.array([np.abs(np.fft.rfft(f))**2 for f in frames]).T
    freq_bins = np.linspace(0, sr/2, n_fft//2 + 1)
    mel_low  = 2595 * np.log10(1 + 80   / 700)
    mel_high = 2595 * np.log10(1 + 8000 / 700)
    mel_pts  = np.linspace(mel_low, mel_high, n_mels + 2)
    hz_pts   = 700 * (10 ** (mel_pts / 2595) - 1)
    fb = np.zeros((n_mels, n_fft//2 + 1))
    for m in range(1, n_mels + 1):
        f0, f1, f2 = hz_pts[m-1], hz_pts[m], hz_pts[m+1]
        for k, f in enumerate(freq_bins):
            if f0 <= f < f1:  fb[m-1, k] = (f - f0) / (f1 - f0 + 1e-12)
            elif f1 <= f <= f2: fb[m-1, k] = (f2 - f) / (f2 - f1 + 1e-12)
    return np.log(fb @ specs + 1e-9)

# yes：高频辅音 [3500, 4200] + 中频元音 [500, 900]
yes_sig    = synth_keyword([3500, 4200, 500,  900], [0.6, 0.4, 0.8, 0.5])
# no：低频鼻音 [200, 400] + 中频元音 [600, 1000]
no_sig     = synth_keyword([200,  400,  600, 1000], [0.7, 0.5, 0.8, 0.4])
# 误分类的 no 样本：鼻音极弱，高频噪声强，让模型误判为 yes
no_bad_sig = synth_keyword([200,  4000, 500,  900], [0.1, 0.5, 0.7, 0.5])

mel_yes    = naive_mel(yes_sig)
mel_no     = naive_mel(no_sig)
mel_no_bad = naive_mel(no_bad_sig)

fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
items = [
    (mel_yes,    'yes 正确样本',         '#2980b9'),
    (mel_no,     'no 正确样本',           '#27ae60'),
    (mel_no_bad, 'no 被误判为 yes\n（辅音区与 yes 相似）', '#e74c3c'),
]
for ax, (mel, title, bc) in zip(axes, items):
    im = ax.imshow(mel, aspect='auto', origin='lower',
                   cmap='magma', interpolation='nearest')
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('时间帧', fontsize=9)
    ax.set_ylabel('Mel 滤波器索引', fontsize=9)
    for spine in ax.spines.values():
        spine.set_edgecolor(bc); spine.set_linewidth(2.5)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.suptitle(
    f'Mel 图对比  (n_mels={N_MELS}, hop={HOP})  — 红框=误分类样本',
    fontsize=12, y=1.02
)
plt.tight_layout()
plt.show()

print('✅ 观察要点：')
print('  · 正确 yes（蓝）：高频滤波器（上方）有明显能量条带')
print('  · 正确 no（绿）：低频滤波器（下方）有鼻音能量')
print('  · 误分类 no（红）：低频鼻音消失，高频条带出现 → 与 yes 高度相似')
print(f'  · 将 N_MELS 从 {N_MELS} 改为 80，可观察频率分辨率提升对区分度的影响')

## 本课收束

本节用 `naive_mel()`（手写三角滤波器）生成 mel 谱，用 matplotlib DAG 图可视化 `Value` 节点的 `data`/`grad`，并用合成信号演示了 train/val loss 曲线的过拟合形态与 early stopping 时机。三张图直接连接 Aurora 的核心模块：`aurora.audio.mel`（mel 滤波器组）提供真实特征，`L54_value_autograd` 的 `Value.backward()` 填入真实梯度，`L58_training_loop` 的训练循环输出真实 loss 曲线。下一节（L66）将进入 ASR 系统全览：了解 CTC、Attention-based Encoder-Decoder 和 Whisper 三类主流架构，以及 Aurora 语音模块的整体设计思路。

In [ ]:
# ── 独立数学断言：验证训练可视化核心性质（无需学生实现）────────────────────────
import numpy as np

# 1. early_stopping 不依赖实现：手写参考版验证逻辑
def _ref_early_stopping(val_losses, patience):
    best_loss, best_ep, no_imp = float('inf'), 1, 0
    for ep, loss in enumerate(val_losses, 1):
        if loss < best_loss:
            best_loss, best_ep, no_imp = loss, ep, 0
        else:
            no_imp += 1
            if no_imp >= patience:
                return best_ep
    return best_ep

cases = [
    ([0.9, 0.7, 0.5, 0.6, 0.65, 0.7, 0.72, 0.75], 3, 3),
    ([0.9, 0.8, 0.7, 0.6, 0.5, 0.4],               2, 6),
    ([1.0, 0.5, 0.6, 0.7, 0.8, 0.9],               2, 2),
]
for losses, pat, expected in cases:
    got = _ref_early_stopping(losses, pat)
    assert got == expected, f"losses={losses},pat={pat}: got={got},exp={expected}"
print(f"1 ✅  early_stopping 参考实现：3组测试全部通过（patience=3→ep3, 2→ep6, 2→ep2）")

# 2. 混淆矩阵对角线 = 正确预测数，行归一化 = recall
rng = np.random.default_rng(7)
n_cls = 4
y_true_q = rng.integers(0, n_cls, 100)
y_pred_q = y_true_q.copy()
# 故意误分类一些
mask = rng.random(100) < 0.2
y_pred_q[mask] = rng.integers(0, n_cls, mask.sum())

CM_q = np.zeros((n_cls, n_cls), dtype=int)
for t, p in zip(y_true_q, y_pred_q): CM_q[t, p] += 1

acc_from_diag = CM_q.diagonal().sum() / CM_q.sum()
acc_direct = (y_true_q == y_pred_q).mean()
assert abs(acc_from_diag - acc_direct) < 1e-12
print(f"2 ✅  CM对角线/总数 = 准确率 = {acc_from_diag:.4f}（与直接计算一致）")

# 3. 行归一化 = recall per class
CM_norm_q = CM_q.astype(float) / (CM_q.sum(axis=1, keepdims=True) + 1e-9)
recall_q = CM_q.diagonal() / (CM_q.sum(axis=1) + 1e-9)
assert np.allclose(CM_norm_q.diagonal(), recall_q, atol=1e-10)
print(f"3 ✅  CM行归一化对角线 = recall per class: {np.round(recall_q,3)}")

# 4. 泛化间隔（generalization gap）= val_loss - train_loss > 0（过拟合阶段）
train_l = np.array([1.0, 0.7, 0.5, 0.4, 0.35, 0.3])
val_l   = np.array([1.0, 0.72, 0.55, 0.5, 0.6, 0.75])
gap = val_l - train_l
overfit_start = np.argmax(gap > 0.05) + 1   # 第一个 gap>0.05 的 epoch
assert overfit_start >= 1
print(f"4 ✅  泛化间隔 gap={np.round(gap,3)}, 过拟合开始于epoch≥{overfit_start}")

# 5. 梯度范数 L2 = sqrt(sum(g²))，验证公式
import torch, torch.nn as nn
g_test = torch.tensor([3.0, 4.0])
norm_torch = g_test.norm().item()
norm_manual = float(np.sqrt((3**2 + 4**2)))
assert abs(norm_torch - norm_manual) < 1e-6
print(f"5 ✅  梯度L2范数: ||[3,4]||₂ = √(9+16) = {norm_manual:.4f}（torch={norm_torch:.4f}）")


---

→ **下一课**　[L66 · ASR 系统全览](../7_asr/L66_asr_overview.ipynb)

> 下节课将学习 **ASR 系统全览**：声学模型 → 语言模型 → 解码器，端到端 vs 经典流水线。